# Paper 60 Implementation: STEVE vs Picket Fence Spectroscopy / STEVE 대 Picket Fence 분광 구현

**Reference**: Gillies, D. M., et al. (2019). *First observations from the TREx Spectrograph: The optical spectrum of STEVE and the Picket Fence phenomena.* Geophysical Research Letters, 46(13), 7207-7213. https://doi.org/10.1029/2019GL083272

## Goal / 목표

Reproduce the qualitative findings of Gillies et al. (2019) using a synthetic spectroscopic model:

1. **STEVE forward spectrum** — broadband continuum (NO+O chemiluminescence shape) plus an enhanced OI 630.0 nm red line driven by a thermal Maxwellian tail at elevated $T_e$.
2. **Picket Fence forward spectrum** — auroral spectrum dominated by OI 557.7 nm (with weak 427.8 nm and OH Meinel band) from electron precipitation.
3. **Differential spectroscopy** — show that the N$_2^+$ 427.8 nm line is enhanced in the Picket Fence but NOT in STEVE, reproducing the paper's key argument.
4. **Joule heating estimate** — compute the SAID frictional heating power per unit volume that elevates $T_e$ in the STEVE region.
5. **Picket Fence vertical structure** — toy model of inertial Alfvén-wave-driven precipitation creating narrow vertical pillars.

본 노트북은 Gillies et al. (2019) 논문의 결론을 합성 분광 모델로 정성적으로 재현한다. 실제 TREx 데이터에 접근할 수 있다면 절대 보정·keogram 추출 단계를 추가하면 된다 (data link in paper acknowledgments).

This notebook qualitatively reproduces the paper's conclusions with a synthetic spectroscopic model. Real TREx data ingestion can be added if the public dataset is downloaded.

In [ ]:
"""Imports and global plotting settings for the paper-60 implementation."""

from __future__ import annotations

import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import Sequence

# Constants in CGS unless otherwise marked.
K_B_EV = 8.617333e-5   # Boltzmann constant in eV/K
K_B_CGS = 1.380649e-16  # erg/K
M_O_CGS = 16 * 1.66054e-24  # mass of atomic oxygen in grams
EV_TO_ERG = 1.60218e-12     # eV to erg

plt.rcParams.update({
    "figure.figsize": (10, 5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
})

## 1. Synthetic Spectrum Generator / 합성 스펙트럼 생성기

We build a spectrum on the TREx wavelength grid (384-804 nm at 0.41 nm/pixel) as the sum of:

- A *continuum* component shaped after the NO+O chemiluminescent nightglow (broad bell ~400-700 nm).
- A set of Gaussian *emission lines* (427.8, 557.7, 630.0 nm, plus a low-amplitude OH Meinel band band-head near 731 nm).
- A flat *background* representing instrumental and stellar leakage.

STEVE / Picket Fence / background pixel are produced with different mixing weights, mirroring the paper's Table 1.

TREx 파장 격자(384-804 nm, 0.41 nm/pixel) 위에서 (i) NO+O 화학발광형 연속체, (ii) 가우시안 방출선 묶음(427.8, 557.7, 630.0 nm + OH Meinel band 731 nm), (iii) 평탄 배경의 합으로 합성 스펙트럼을 만든다. STEVE / Picket Fence / 배경 픽셀은 가중치만 다르게 한다.

In [ ]:
@dataclass(frozen=True)
class EmissionLine:
    """Gaussian emission line specification.

    Attributes:
        center_nm: Central wavelength in nanometers.
        amplitude_R_per_nm: Peak amplitude in rayleighs per nanometer.
        fwhm_nm: Full width at half maximum in nanometers.
        label: Human-readable label, e.g. 'OI 557.7 nm'.
    """
    center_nm: float
    amplitude_R_per_nm: float
    fwhm_nm: float
    label: str


def gaussian(x: np.ndarray, center: float, amplitude: float, fwhm: float) -> np.ndarray:
    """Return a Gaussian profile evaluated on x.

    Args:
        x: Input grid (e.g. wavelength array in nm).
        center: Gaussian center on the same units as x.
        amplitude: Peak value of the Gaussian.
        fwhm: Full width at half maximum on the same units as x.

    Returns:
        Gaussian evaluated on x with the requested peak and width.
    """
    sigma = fwhm / (2.0 * np.sqrt(2.0 * np.log(2.0)))
    return amplitude * np.exp(-0.5 * ((x - center) / sigma) ** 2)


def no_continuum_shape(wavelength_nm: np.ndarray, peak_nm: float = 560.0,
                       width_nm: float = 250.0) -> np.ndarray:
    """Return the shape of the NO+O chemiluminescent nightglow continuum.

    Modeled as a broad smooth bell in 400-700 nm. The shape is normalized to
    a peak value of 1; brightness scaling is applied by the caller.

    Args:
        wavelength_nm: Wavelength grid in nanometers.
        peak_nm: Continuum peak position (literature ~560-580 nm).
        width_nm: Approximate FWHM-equivalent of the broad bell.

    Returns:
        Dimensionless continuum profile peaked near `peak_nm`.
    """
    sigma = width_nm / (2.0 * np.sqrt(2.0 * np.log(2.0)))
    return np.exp(-0.5 * ((wavelength_nm - peak_nm) / sigma) ** 2)


def build_spectrum(wavelength_nm: np.ndarray,
                   continuum_amplitude: float,
                   lines: Sequence[EmissionLine],
                   flat_background: float = 0.0) -> np.ndarray:
    """Build a synthetic spectrum as continuum + Gaussian lines + flat offset.

    Args:
        wavelength_nm: Wavelength grid (nm).
        continuum_amplitude: Peak rayleighs/nm of the NO+O-like continuum.
        lines: Iterable of EmissionLine specifications.
        flat_background: Constant background offset in rayleighs/nm.

    Returns:
        Spectrum in rayleighs/nm on the input wavelength grid.
    """
    spectrum = continuum_amplitude * no_continuum_shape(wavelength_nm)
    for ln in lines:
        spectrum = spectrum + gaussian(wavelength_nm, ln.center_nm,
                                       ln.amplitude_R_per_nm, ln.fwhm_nm)
    spectrum = spectrum + flat_background
    return spectrum

## 2. STEVE, Picket Fence, and Background Spectra / STEVE·Picket Fence·배경 스펙트럼

We assemble three pixels — STEVE, Picket Fence, and background airglow — using line amplitudes/continuum strengths chosen to roughly reproduce the brightness ordering reported in Table 1 of the paper.

Table 1의 광도 비율을 대략 재현하도록 STEVE·Picket Fence·background 세 픽셀의 라인/연속체 진폭을 설정한다.

In [ ]:
# TREx-like wavelength grid: 384.1 - 804 nm at 0.41 nm/pixel.
wavelength_nm = np.arange(384.1, 804.0, 0.41)

# Auroral / airglow lines on a common width (instrument-resolution dominated).
FWHM_NM = 0.9  # ~2 spectral pixels after smoothing

# Background airglow: weak 557.7, weak 630.0, very weak NO+O continuum.
background_lines = [
    EmissionLine(427.8, 1.0, FWHM_NM, "N2+ 427.8 nm"),
    EmissionLine(557.7, 50.0, FWHM_NM, "OI 557.7 nm"),
    EmissionLine(630.0, 25.0, FWHM_NM, "OI 630.0 nm"),
    EmissionLine(731.0, 4.0, 8.0, "OH Meinel band"),
]
background_spectrum = build_spectrum(wavelength_nm,
                                     continuum_amplitude=12.0,
                                     lines=background_lines,
                                     flat_background=2.0)

# STEVE: continuum strongly elevated, 630.0 nm enhanced, 427.8 and 557.7 unchanged from background.
steve_lines = [
    EmissionLine(427.8, 1.0, FWHM_NM, "N2+ 427.8 nm"),    # NOT enhanced - paper key point
    EmissionLine(557.7, 50.0, FWHM_NM, "OI 557.7 nm"),    # not enhanced
    EmissionLine(630.0, 75.0, FWHM_NM, "OI 630.0 nm"),    # enhanced (3x background)
    EmissionLine(731.0, 4.0, 8.0, "OH Meinel band"),
]
steve_spectrum = build_spectrum(wavelength_nm,
                                continuum_amplitude=35.0,  # large continuum bump
                                lines=steve_lines,
                                flat_background=2.0)

# Picket Fence: 557.7 nm strongly enhanced (~5x), small OH increase, no continuum.
picket_lines = [
    EmissionLine(427.8, 1.0, FWHM_NM, "N2+ 427.8 nm"),
    EmissionLine(557.7, 250.0, FWHM_NM, "OI 557.7 nm"),   # 5x background
    EmissionLine(630.0, 25.0, FWHM_NM, "OI 630.0 nm"),
    EmissionLine(731.0, 8.0, 8.0, "OH Meinel band"),       # weak enhancement
]
picket_spectrum = build_spectrum(wavelength_nm,
                                 continuum_amplitude=12.0,
                                 lines=picket_lines,
                                 flat_background=2.0)

fig, ax = plt.subplots()
ax.plot(wavelength_nm, steve_spectrum, color="royalblue", lw=1.4, label="STEVE")
ax.plot(wavelength_nm, picket_spectrum, color="forestgreen", lw=1.4, alpha=0.8, label="Picket Fence")
ax.plot(wavelength_nm, background_spectrum, color="crimson", lw=1.0, alpha=0.6, label="Background airglow")
for ln_center, label in [(427.8, "N2+"), (557.7, "OI 557.7"), (630.0, "OI 630.0")]:
    ax.axvline(ln_center, ls=":", color="k", alpha=0.4)
    ax.text(ln_center + 1.0, ax.get_ylim()[1] * 0.85 if ax.get_ylim()[1] > 0 else 0, label, fontsize=8)
ax.set_xlabel("Wavelength [nm]")
ax.set_ylabel("Brightness [R/nm]")
ax.set_title("Synthetic spectra: STEVE vs Picket Fence vs background (after Gillies et al. 2019)")
ax.legend()
plt.tight_layout()
plt.show()

## 3. Differential Spectroscopy / 차분 분광

Following the paper, the **STEVE pixel minus background** spectrum should reveal a broadband continuum + 630.0 nm bump but NO 427.8 nm or 557.7 nm enhancement. The **Picket Fence pixel minus background** should be dominated by a sharp 557.7 nm spike, with no continuum.

논문대로, (STEVE − background)는 광역 연속체 + 630 nm 봉우리만 남고, (Picket − background)는 557.7 nm 큰 봉우리만 남아야 한다.

In [ ]:
steve_minus_bg = steve_spectrum - background_spectrum
picket_minus_bg = picket_spectrum - background_spectrum

fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

axes[0].plot(wavelength_nm, steve_minus_bg, color="royalblue", lw=1.4)
axes[0].axhline(0, color="k", lw=0.5)
axes[0].set_ylabel("STEVE - bg [R/nm]")
axes[0].set_title("Differential spectrum: STEVE - background (continuum + 630 nm; NO 427.8 nm)")

axes[1].plot(wavelength_nm, picket_minus_bg, color="forestgreen", lw=1.4)
axes[1].axhline(0, color="k", lw=0.5)
axes[1].set_ylabel("Picket - bg [R/nm]")
axes[1].set_xlabel("Wavelength [nm]")
axes[1].set_title("Differential spectrum: Picket Fence - background (dominated by 557.7 nm)")

for ax in axes:
    for ln_center, label in [(427.8, "N2+"), (557.7, "OI 557.7"), (630.0, "OI 630.0")]:
        ax.axvline(ln_center, ls=":", color="k", alpha=0.4)

plt.tight_layout()
plt.show()

# Print integrated quantities for comparison with Table 1.
def integrate_band(wave_nm: np.ndarray, spec: np.ndarray,
                   wave_min: float, wave_max: float) -> float:
    """Trapezoidal integral of spec over [wave_min, wave_max] in rayleighs.

    Args:
        wave_nm: Wavelength grid in nanometers.
        spec: Brightness spectrum in rayleighs/nm.
        wave_min: Lower wavelength bound (nm).
        wave_max: Upper wavelength bound (nm).

    Returns:
        Band-integrated brightness in rayleighs.
    """
    mask = (wave_nm >= wave_min) & (wave_nm <= wave_max)
    return float(np.trapz(spec[mask], wave_nm[mask]))

I_steve_total = integrate_band(wavelength_nm, steve_spectrum, 400, 700)
I_bg_total = integrate_band(wavelength_nm, background_spectrum, 400, 700)
I_picket_5577 = integrate_band(wavelength_nm, picket_spectrum, 555, 560)
I_bg_5577 = integrate_band(wavelength_nm, background_spectrum, 555, 560)

print(f"STEVE 400-700 nm integrated     : {I_steve_total:8.0f} R")
print(f"Background 400-700 nm integrated: {I_bg_total:8.0f} R")
print(f"STEVE excess above background    : {I_steve_total - I_bg_total:8.0f} R   (paper: ~6,200 R)")
print(f"Picket 557.7 nm band            : {I_picket_5577:8.0f} R")
print(f"Background 557.7 nm band        : {I_bg_5577:8.0f} R")
print(f"Picket green-line excess        : {I_picket_5577 - I_bg_5577:8.0f} R   (paper: ~1,930 R)")

## 4. Thermal Red-Line Excitation by Hot Maxwellian Tail / 고온 Maxwell tail에 의한 적색선 열적 여기

We compute the fraction of Maxwell-Boltzmann electrons with energy above the O($^1$D) threshold (1.96 eV) as a function of $T_e$. This explains why STEVE — sitting in a region of magnetospheric heat flux + SAID Joule heating — shows enhanced 630.0 nm emission *without* particle precipitation.

$$ f_{\geq E_c}(T_e) = \frac{2}{\sqrt{\pi}}\int_{x_c}^{\infty}\!\sqrt{x}\,e^{-x}\,dx,\quad x_c=\frac{E_c}{k_B T_e} $$

Maxwell-Boltzmann 전자 중 1.96 eV 이상인 분율을 $T_e$의 함수로 계산하여 SAR-arc 메커니즘을 정량화.

In [ ]:
from scipy.special import gammaincc


def fraction_above_threshold(T_e_K: np.ndarray, E_threshold_eV: float) -> np.ndarray:
    """Maxwell-Boltzmann fraction with energy above E_threshold.

    Uses the regularized upper incomplete gamma function, since
    f = (2/sqrt(pi)) * integral_{x_c}^{inf} sqrt(x) e^-x dx = Q(3/2, x_c).

    Args:
        T_e_K: Electron temperature(s) in kelvin.
        E_threshold_eV: Energy threshold in electronvolts.

    Returns:
        Fraction of electrons with E >= E_threshold.
    """
    x_c = E_threshold_eV / (K_B_EV * T_e_K)
    return gammaincc(1.5, x_c)


T_e_grid = np.linspace(1000, 10000, 200)
frac_OD = fraction_above_threshold(T_e_grid, 1.96)

fig, ax = plt.subplots()
ax.semilogy(T_e_grid, frac_OD, color="darkred", lw=1.6)
ax.axvline(3000, ls=":", color="k")
ax.axvline(6000, ls=":", color="k")
ax.set_xlabel("Electron temperature T_e [K]")
ax.set_ylabel("Fraction of electrons with E >= 1.96 eV")
ax.set_title("Maxwellian high-energy tail driving STEVE 630.0 nm thermal emission")
for T_query in [3000, 6000, 8000]:
    val = float(fraction_above_threshold(np.array([T_query]), 1.96)[0])
    print(f"T_e = {T_query:5d} K  ->  fraction(>=1.96 eV) = {val:.3e}")
plt.tight_layout()
plt.show()

## 5. Joule Heating from SAID-Driven Frictional Drift / SAID 마찰 가열

Inside STEVE, ions in the SAID jet move at $\sim$1-3 km/s relative to the neutrals. The volumetric heating rate is

$$ Q_J \approx n_i\,m_i\,\nu_{in}\,|\mathbf{v}_i-\mathbf{v}_n|^2 $$

We sweep the relative velocity and the F-region collision frequency to bracket plausible STEVE conditions.

STEVE 영역에서 이온이 SAID 흐름에 따라 중성 대기 대비 1-3 km/s로 운동, 부피당 가열률을 추정한다.

In [ ]:
def joule_heating_rate(n_i_cm3: float, v_rel_km_s: float, nu_in_per_s: float,
                      ion_mass_g: float = M_O_CGS) -> float:
    """Volumetric Joule heating rate from frictional ion-neutral drift.

    Q_J = n_i * m_i * nu_in * |v_i - v_n|^2  (CGS, erg cm^-3 s^-1).

    Args:
        n_i_cm3: Ion density (cm^-3).
        v_rel_km_s: Ion-neutral relative drift speed (km/s).
        nu_in_per_s: Ion-neutral collision frequency (s^-1).
        ion_mass_g: Ion mass in grams (default: O+).

    Returns:
        Volumetric heating rate in erg cm^-3 s^-1.
    """
    v_rel_cm_s = v_rel_km_s * 1.0e5
    return n_i_cm3 * ion_mass_g * nu_in_per_s * v_rel_cm_s ** 2


v_rel_grid = np.linspace(0.5, 4.0, 60)        # km/s
nu_in_grid = np.array([0.3, 1.0, 3.0])         # s^-1 across F-region altitudes
n_i = 1.0e5                                     # cm^-3 typical F-region

fig, ax = plt.subplots()
for nu in nu_in_grid:
    Q = np.array([joule_heating_rate(n_i, v, nu) for v in v_rel_grid])
    ax.plot(v_rel_grid, Q, lw=1.4, label=f"nu_in = {nu} s^-1")
ax.set_yscale("log")
ax.set_xlabel("|v_i - v_n|  [km/s]")
ax.set_ylabel("Joule heating rate Q_J [erg cm^-3 s^-1]")
ax.set_title("Frictional heating in the SAID/STEVE region (n_i = 1e5 cm^-3, O+)")
ax.legend()
plt.tight_layout()
plt.show()

Q_demo = joule_heating_rate(n_i_cm3=1.0e5, v_rel_km_s=2.0, nu_in_per_s=1.0)
print(f"Demo Q_J at v=2 km/s, nu=1 s^-1 : {Q_demo:.2e} erg cm^-3 s^-1")
print("For reference, this is enough to raise T_e by thousands of K when balanced "
      "against radiative+collisional losses (Liang et al. 2017).")

## 6. Picket Fence Vertical Pillars — Toy Alfvén-Wave Model / Picket Fence 수직 기둥 토이 모델

We model the Picket Fence morphology as a periodic horizontal pattern of precipitation amplitude, multiplied by a vertical altitude profile peaked near 110 km (where 557.7 nm production maximizes).

Picket Fence 형태를 수평 주기 강수 패턴 × 110 km 부근 고도 프로파일 곱으로 모델링한다.

In [ ]:
def vertical_557_profile(z_km: np.ndarray, peak_km: float = 110.0,
                          width_km: float = 8.0) -> np.ndarray:
    """Vertical 557.7 nm volume emission profile, Gaussian shape.

    Args:
        z_km: Altitude grid (km).
        peak_km: Altitude of peak emission (typical green-line peak).
        width_km: Gaussian sigma in km.

    Returns:
        Normalized vertical profile, peak = 1.
    """
    return np.exp(-0.5 * ((z_km - peak_km) / width_km) ** 2)


def picket_horizontal_pattern(x_km: np.ndarray, lambda_h_km: float = 12.0,
                              duty: float = 0.35) -> np.ndarray:
    """Horizontal precipitation pattern: smoothed square wave (pillars).

    Args:
        x_km: Horizontal coordinate (km).
        lambda_h_km: Horizontal wavelength of the pillars.
        duty: Fractional width of the bright pillar.

    Returns:
        Horizontal profile in [0, 1].
    """
    phase = (x_km % lambda_h_km) / lambda_h_km
    raw = np.where(phase < duty, 1.0, 0.0)
    # smooth pillars to mimic finite spread
    kernel_size = 5
    kernel = np.ones(kernel_size) / kernel_size
    smoothed = np.convolve(raw, kernel, mode="same")
    return smoothed


x_km = np.linspace(0, 80, 400)
z_km = np.linspace(80, 200, 250)
X, Z = np.meshgrid(x_km, z_km)

horizontal = picket_horizontal_pattern(X, lambda_h_km=12.0, duty=0.35)
vertical = vertical_557_profile(Z, peak_km=110.0, width_km=8.0)
emission_557 = horizontal * vertical * 250.0  # peak ~ R/nm units

fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(emission_557, origin="lower", aspect="auto",
               extent=[x_km.min(), x_km.max(), z_km.min(), z_km.max()],
               cmap="viridis")
cbar = plt.colorbar(im, ax=ax, label="557.7 nm volume emission [R/nm]")
ax.set_xlabel("Horizontal distance [km]")
ax.set_ylabel("Altitude [km]")
ax.set_title("Toy Picket Fence: vertical 557.7 nm pillars (Alfven-wave-driven precipitation)")
plt.tight_layout()
plt.show()

## 7. Summary / 요약

1. **STEVE differential spectrum** reproduces the paper's result: continuum + 630.0 nm enhancement, **no** N$_2^+$ 427.8 nm enhancement → no precipitation → STEVE is not aurora.
2. **Picket Fence differential spectrum** is dominated by a 5x 557.7 nm spike → consistent with electron precipitation aurora.
3. **Maxwell-tail calculation** shows that going from $T_e=3{,}000$ K to $T_e=6{,}000$ K increases the >1.96 eV electron fraction by tens of times — the SAR-arc-style mechanism for STEVE's red-line component.
4. **Joule-heating estimate** shows that even modest SAID drift speeds (~2 km/s) provide $\sim10^{-7}$ erg cm$^{-3}$ s$^{-1}$ of volumetric heating — abundant power to elevate $T_e$.
5. **Toy Picket Fence** model captures the narrow vertical pillars at $\sim$110 km matching the OI 557.7 nm emission peak.

본 노트북은 합성 데이터 기반 정성적 재현이며, 실제 TREx 분광 데이터를 받아 (i) Hg/He 보정 → 파장 등록, (ii) flat-field/dark 처리, (iii) keogram 생성, (iv) Gaussian fit으로 STEVE/Picket 위치 추출 → background 차분 단계를 정량적으로 추가하면 논문 Figure 1, Figure 2를 그대로 재현할 수 있다.